In [1]:
import sys
from pathlib import Path

# Garante que o pacote do projeto (src/spark_delta) seja encontrado mesmo que o
# kernel não tenha sido instalado com `uv sync` (fallback para src/).
try:
    import spark_delta  # noqa: F401
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))
    sys.path.insert(0, str(Path.cwd().resolve() / "src"))

import pandas as pd

from spark_delta.config import DATA_DIR, settings
from spark_delta.sqlserver import connect

print(f"Servidor: {settings.db_server}:{settings.db_port}")
print(f"Database: {settings.db_database}")
print(f"CSVs em:  {DATA_DIR}")

Servidor: localhost:1433
Database: LojaDB
CSVs em:  /home/maicou/projeto2/spark-delta/data


In [2]:
# Conecta no master para criar o database
conn_master = connect()
cur = conn_master.cursor()
cur.execute(f"""
    IF NOT EXISTS (SELECT name FROM sys.databases WHERE name = '{settings.db_database}')
        CREATE DATABASE [{settings.db_database}]
""")
cur.close()
conn_master.close()
print(f"Database [{settings.db_database}] criado/verificado!")

conn = connect(settings.db_database)
cursor = conn.cursor()
print(f"Conectado ao [{settings.db_database}]")

Database [LojaDB] criado/verificado!
Conectado ao [LojaDB]


In [3]:
ddl = {
    "categoria": """
        CREATE TABLE categoria (
            cd_categoria  INT          NOT NULL PRIMARY KEY,
            nm_categoria  VARCHAR(50)  NOT NULL
        )""",
    "fornecedor": """
        CREATE TABLE fornecedor (
            cd_fornecedor INT          NOT NULL PRIMARY KEY,
            nm_fornecedor VARCHAR(100) NOT NULL,
            cnpj          CHAR(14)     NOT NULL,
            cidade        VARCHAR(60)  NOT NULL,
            uf            CHAR(2)      NOT NULL
        )""",
    "produto": """
        CREATE TABLE produto (
            cd_produto    INT           NOT NULL PRIMARY KEY,
            nm_produto    VARCHAR(100)  NOT NULL,
            cd_categoria  INT           NOT NULL REFERENCES categoria (cd_categoria),
            cd_fornecedor INT           NOT NULL REFERENCES fornecedor (cd_fornecedor),
            preco         DECIMAL(12,2) NOT NULL,
            dt_cadastro   DATE          NOT NULL
        )""",
    "estoque": """
        CREATE TABLE estoque (
            cd_produto     INT          NOT NULL PRIMARY KEY REFERENCES produto (cd_produto),
            quantidade     INT          NOT NULL,
            estoque_minimo INT          NOT NULL,
            dt_atualizacao DATETIME2(0) NOT NULL
        )""",
    "cliente": """
        CREATE TABLE cliente (
            cd_cliente  INT          NOT NULL PRIMARY KEY,
            nm_cliente  VARCHAR(100) NOT NULL,
            cpf         CHAR(11)     NOT NULL,
            email       VARCHAR(100) NOT NULL,
            cidade      VARCHAR(60)  NOT NULL,
            uf          CHAR(2)      NOT NULL,
            dt_cadastro DATE         NOT NULL
        )""",
    "venda": """
        CREATE TABLE venda (
            cd_venda        INT           NOT NULL PRIMARY KEY,
            cd_cliente      INT           NOT NULL REFERENCES cliente (cd_cliente),
            dt_venda        DATETIME2(0)  NOT NULL,
            forma_pagamento VARCHAR(20)   NOT NULL,
            vl_total        DECIMAL(14,2) NOT NULL
        )""",
    "item_venda": """
        CREATE TABLE item_venda (
            cd_venda    INT           NOT NULL REFERENCES venda (cd_venda),
            cd_produto  INT           NOT NULL REFERENCES produto (cd_produto),
            quantidade  INT           NOT NULL,
            vl_unitario DECIMAL(12,2) NOT NULL,
            PRIMARY KEY (cd_venda, cd_produto)
        )""",
}

for tabela, comando in ddl.items():
    cursor.execute(f"IF OBJECT_ID('dbo.{tabela}', 'U') IS NULL {comando}")
    print(f"  tabela {tabela} criada/verificada")

  tabela categoria criada/verificada
  tabela fornecedor criada/verificada
  tabela produto criada/verificada
  tabela estoque criada/verificada
  tabela cliente criada/verificada
  tabela venda criada/verificada
  tabela item_venda criada/verificada


In [4]:
tabelas = list(ddl)  # mesma ordem do DDL (respeita as FKs)

for tabela in tabelas:
    cursor.execute(f"SELECT COUNT(*) FROM {tabela}")
    existentes = cursor.fetchone()[0]
    if existentes > 0:
        print(f"  {tabela:<12} já possui {existentes} registros, pulando")
        continue

    # Tudo como texto: preserva zeros à esquerda (CPF/CNPJ) e o SQL Server
    # converte para INT/DECIMAL/DATE na inserção
    df = pd.read_csv(DATA_DIR / f"{tabela}.csv", dtype=str, keep_default_na=False)
    colunas = ", ".join(df.columns)
    params = ", ".join("?" for _ in df.columns)
    linhas = [tuple(v if v != "" else None for v in row) for row in df.itertuples(index=False)]

    for i in range(0, len(linhas), 2000):
        cursor.executemany(f"INSERT INTO {tabela} ({colunas}) VALUES ({params})", linhas[i:i + 2000])
    print(f"  {tabela:<12} {len(linhas):>6} registros inseridos")

print("\nCarga concluída!")

  categoria         8 registros inseridos
  fornecedor       25 registros inseridos
  produto         120 registros inseridos
  estoque         120 registros inseridos
  cliente        1000 registros inseridos
  venda          5000 registros inseridos
  item_venda    10964 registros inseridos

Carga concluída!


In [5]:
print(f"{'Tabela':<12} {'Registros':>10}")
print("-" * 23)
for tabela in tabelas:
    cursor.execute(f"SELECT COUNT(*) FROM {tabela}")
    print(f"{tabela:<12} {cursor.fetchone()[0]:>10}")

Tabela        Registros
-----------------------
categoria             8
fornecedor           25
produto             120
estoque             120
cliente            1000
venda              5000
item_venda        10964


In [6]:
# Amostra com JOIN: produto + categoria + estoque
cursor.execute("""
    SELECT TOP 10 p.cd_produto, p.nm_produto, c.nm_categoria, p.preco, e.quantidade
    FROM produto p
    JOIN categoria c ON c.cd_categoria = p.cd_categoria
    JOIN estoque   e ON e.cd_produto   = p.cd_produto
    ORDER BY p.cd_produto
""")
pd.DataFrame.from_records(cursor.fetchall(), columns=[d[0] for d in cursor.description])

,cd_produto,nm_produto,nm_categoria,preco,quantidade
0,1,Notebook 14 pol i5 Asus,Computadores,8455.52,44
1,2,Notebook 14 pol i5 Acer,Computadores,5928.86,112
2,3,Notebook 14 pol i5 HP,Computadores,9211.51,13
3,4,Notebook 15 pol i7 Acer,Computadores,4766.00,32
4,5,Notebook 15 pol i7 Asus,Computadores,5803.39,87
5,6,Notebook 15 pol i7 Dell,Computadores,2338.80,79
6,7,Notebook gamer RTX Dell,Computadores,6935.08,89
7,8,Notebook gamer RTX Acer,Computadores,3277.17,86
8,9,Notebook gamer RTX HP,Computadores,4117.31,38
9,10,Ultrabook 13 pol HP,Computadores,5380.41,27


In [7]:
cursor.close()
conn.close()
print("Conexão encerrada.")

Conexão encerrada.
